# demo_impl — 실시간 필터(impl) 모델: 데이터 · 파인튜닝 · 채점 · 응답 시간

**impl 의 과제**: 채팅 **문장 1개**를 받아 ① 숨겨야 하는 욕설 조각, ② 비난·패드립·성희롱·정치질 여부, ③ 그 안에 게임 피드백이 있는지를 판별한다.
화면에는 원본과 숨김 문장을 보여 준다. 모든 채팅에 실행되므로 **0.5초 미만(최대 1초 미만)** 이어야 한다.

| 무엇이 정하나 | 내용 |
|---|---|
| 규칙 (`lexicon.py`, `impl.py`) | 가릴 욕설 조각, 피드백 조각, 대상, 등급(L0~L3) |
| 모델 (이 노트북에서 파인튜닝) | 유형 점수 5개: 패드립 · 성희롱 · 비난 · 정치질 · 협박 (욕설은 사전이 맡는다) |

대화 흐름을 보고 신고·고소를 정하는 것은 judge 의 일이다 → `demo_judge.ipynb`.

실행: 맨 위부터 전체 실행. `RETRAIN = False` 로 두면 학습을 건너뛰고 저장된 모델로 채점만 한다.
모든 문장은 **AI 가 쓴 가상 문장(검수 전)** 이다. 수치는 그만큼 낙관적일 수 있다.

## 0. 설정

In [1]:
import csv
import json
import os
import statistics
import time
import warnings
from collections import Counter
from pathlib import Path

from transformers.utils import logging as hf_logging

from clean_chat.core.config import resolve_repo_path
from clean_chat.modules.analyze import evaluate, impl, model, train

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
FIX = resolve_repo_path("fixtures/analyze")

BASE = "beomi/KcELECTRA-base"  # 사전학습 모델 (문장 1개 = 최대 64토큰)
LR = 5e-5  # 학습률
BATCH_SIZE = 16  # 한 번에 묶어 보는 문장 수
EPOCHS = 20  # 최대 epoch
PATIENCE = 6  # 검증 점수가 이만큼 안 오르면 멈춤
VAL_RATIO = 0.15  # 학습 문장 중 검증용 비율
SEED = 1
RETRAIN = True
OUT_DIR = "models/ft/demo-impl"  # git 제외. 앱이 쓰는 models/analyze-classifier 는 건드리지 않는다

TRAIN_FILES = [FIX / "labeled_parsed.csv", FIX / "train_aug.csv"]
TEST_SETS = {
    "기존 시험 60문장": FIX / "labeled_parsed.csv",
    "챔피언 이름 시험 43문장": FIX / "champion_eval_draft.csv",
    "협박 시험 40문장": FIX / "threat_eval_draft.csv",
}
print("유형:", train.LABELS)

유형: ['profanity', 'family_insult', 'sexual_harassment', 'blame', 'instigation', 'threat']


## 1. 학습에 쓴 데이터

`split=train` 인 행만 학습에 쓴다. `split=test` 는 채점 전용이고 학습·기준 조정에 쓰지 않는다.
협박은 유형 `threat`(ADR-0021)로 학습한다. 게임 밖 현실의 해악을 말하는 문장만 협박이고, "바론 죽인다" 같은 게임 안의 말은 정상이다.

In [2]:
def read_rows(path, split):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return [r | {"file": Path(path).name} for r in csv.DictReader(f) if r["split"] == split]


train_rows = [r for f in TRAIN_FILES for r in read_rows(f, "train")]
used = train.load_examples(TRAIN_FILES)
print(f"학습용 행 {len(train_rows)}개 (그중 실제 학습에 쓰는 것 {len(used)}개)\n")


def batch_of(r):
    if r["file"] == "labeled_parsed.csv":
        return "0차 · 기존 라벨"
    return r["note"].split(" · 검수")[0] if r["note"].startswith("보강") else r["note"]


print("차수별")
for name, n in Counter(batch_of(r) for r in train_rows).most_common():
    print(f"  {n:>4}  {name}")
print("\n등급별 ", dict(sorted(Counter("L" + r["level"] for r in train_rows).items())))
types = Counter(t for r in train_rows for t in r["harm_types"].split("|") if t)
print("유형별 ", dict(types), "· 유형 없음(정상 등)", sum(not r["harm_types"] for r in train_rows))
print(
    "가릴 조각이 있는 문장",
    sum(bool(r["profanity_spans"]) for r in train_rows),
    "· 피드백이 있는 문장",
    sum(r["has_feedback"] == "1" for r in train_rows),
    "· 우회 표기(text ≠ text_norm)",
    sum(r["text"] != r["text_norm"] for r in train_rows),
)

print("\n예시 (등급 | 유형 | 가릴 조각 | 피드백 | 원문)")
for level in "0123":
    for r in [x for x in train_rows if x["level"] == level][:: max(1, len(train_rows) // 12)][:3]:
        print(
            f"  L{r['level']} | {r['harm_types'] or '-':<24} | {r['profanity_spans'] or '-':<8} | {r['has_feedback']} | {r['text']}"
        )

학습용 행 1126개 (그중 실제 학습에 쓰는 것 1126개)

차수별
   370  보강
   335  보강3 · 숨김·피드백·킬뎃
   168  보강2 · 챔피언 이름
   140  0차 · 기존 라벨
    65  보강4 · 협박
    30  보강3 · 말리는 말
    18  보강4 · 정당한 대응

등급별  {'L0': 461, 'L1': 51, 'L2': 482, 'L3': 132}
유형별  {'profanity': 179, 'blame': 439, 'instigation': 67, 'family_insult': 62, 'sexual_harassment': 36, 'threat': 37} · 유형 없음(정상 등) 461
가릴 조각이 있는 문장 196 · 피드백이 있는 문장 148 · 우회 표기(text ≠ text_norm) 91

예시 (등급 | 유형 | 가릴 조각 | 피드백 | 원문)
  L0 | -                        | -        | 0 | 다들 화이팅
  L0 | -                        | -        | 0 | 방금 건 내가 못했어
  L0 | -                        | -        | 0 | 넌 사이드 밀어 우린 미드 있을게
  L1 | profanity                | ㅅㅂ       | 0 | 아 ㅅㅂ 내가 실수함
  L2 | profanity|blame          | ㅂㅅ       | 1 | 야 정글 ㅂㅅ아 용 먼저 치라고
  L2 | blame                    | -        | 0 | 원딜 있으나 마나네
  L2 | blame|instigation        | -        | 0 | 원딜 트롤임 다 같이 신고하자
  L3 | profanity|family_insult  | 니애미|tlqkf | 0 | 탑 니애미 tlqkf
  L3 | threat                   | -        | 

## 2. 채점에 쓴 데이터

세 시험셋 모두 학습 문장과 겹치지 않는다(테스트가 확인). 모두 AI 작성·검수 전이다.
협박 시험셋은 협박 20문장과, 협박처럼 보이는 게임 안의 말 19문장, 경계 1문장이다.

In [3]:
test_rows = {name: evaluate.load_rows(path, "test") for name, path in TEST_SETS.items()}
for name, rows in test_rows.items():
    print(
        f"{name}: {len(rows)}문장 · 등급",
        dict(sorted(Counter("L" + r["level"] for r in rows).items())),
        "· 피드백 있음",
        sum(r.get("has_feedback") == "1" for r in rows),
    )
    for r in rows[:3]:
        print("    ", f"L{r['level']}", r["harm_types"] or "-", "|", r["text"])

기존 시험 60문장: 60문장 · 등급 {'L0': 21, 'L1': 9, 'L2': 21, 'L3': 9} · 피드백 있음 12
     L0 - | 용 30초 뒤에 나와
     L0 - | 정글 블루 줄 수 있어?
     L0 - | 서폿 와드 고마워
챔피언 이름 시험 43문장: 43문장 · 등급 {'L0': 14, 'L1': 2, 'L2': 19, 'L3': 8} · 피드백 있음 6
     L0 - | 이즈 궁 좋았어
     L0 - | 케틀 덫 조심해
     L0 - | 야스오 바람 장막 빠졌어
협박 시험 40문장: 40문장 · 등급 {'L0': 19, 'L2': 1, 'L3': 20} · 피드백 있음 0
     L3 threat | 너 어디 사냐 주소 불러라 찾아감
     L3 threat | 원딜 겜 끝나고 디코로 와라 얼굴 보고 패줄게
     L3 threat | 미드 니 인스타 찾았다 다 뿌려줄게


## 3. 파인튜닝

`train.py` 의 `train()` 을 그대로 부른다. epoch 마다 검증 문장으로 유형별 F1 평균을 재고, 가장 좋았던 epoch 의 가중치를 저장한다.

In [4]:
OUT = resolve_repo_path(OUT_DIR)
history = []
if RETRAIN:
    started = time.perf_counter()
    train.train(
        TRAIN_FILES,
        BASE,
        OUT,
        "demo-impl",
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        seed=SEED,
        val_ratio=VAL_RATIO,
        patience=PATIENCE,
        on_epoch=history.append,
        log=lambda line: print(line.replace(str(OUT), OUT_DIR)),
    )
    print(f"학습 시간 {time.perf_counter() - started:.0f}초")
meta = json.loads((OUT / "meta.json").read_text(encoding="utf-8"))
print({k: meta[k] for k in ("base", "train_rows", "val_rows", "best_epoch", "val_macro_f1", "lr", "seed")})
print("검증 유형별 F1:", meta["val_f1"])

학습 961문장 · 검증 165문장 · mps · 최대 20 epoch · lr 5e-05 · base=beomi/KcELECTRA-base


  epoch 1/20  loss 0.9636  검증 F1 평균 0.124


  epoch 2/20  loss 0.8160  검증 F1 평균 0.622


  epoch 3/20  loss 0.4878  검증 F1 평균 0.675


  epoch 4/20  loss 0.2927  검증 F1 평균 0.723


  epoch 5/20  loss 0.1840  검증 F1 평균 0.747


... (5 outputs left out)

  epoch 11/20  loss 0.0318  검증 F1 평균 0.949


  epoch 12/20  loss 0.0215  검증 F1 평균 0.942


  epoch 13/20  loss 0.0179  검증 F1 평균 0.945  → 조기 종료
  가장 좋은 epoch 7 (검증 F1 평균 0.950) 의 가중치를 저장


저장: models/ft/demo-impl  (평가: uv run python -m clean_chat.modules.analyze.evaluate <CSV> --split test)
학습 시간 129초
{'base': 'beomi/KcELECTRA-base', 'train_rows': 961, 'val_rows': 165, 'best_epoch': 7, 'val_macro_f1': 0.9496, 'lr': 5e-05, 'seed': 1}
검증 유형별 F1: {'profanity': 0.9804, 'family_insult': 0.9412, 'sexual_harassment': 0.8889, 'blame': 0.9457, 'instigation': 0.9412, 'threat': 1.0}


In [5]:
if history:
    top = max(h["loss"] for h in history)
    print(f"{'epoch':>5}  {'학습률':>9}  {'손실':>7}  {'':<20}  {'검증F1':>6}")
    for h in history:
        print(
            f"{h['epoch']:>5}  {h['lr']:>10.2e}  {h['loss']:>8.4f}  {'█' * round(20 * h['loss'] / top):<20}  {h.get('val_f1', 0):>7.3f}  {'●' * round(20 * h.get('val_f1', 0))}"
        )

epoch        학습률       손실                          검증F1
    1    2.50e-05    0.9636  ████████████████████    0.124  ●●
    2    5.00e-05    0.8160  █████████████████       0.622  ●●●●●●●●●●●●
    3    4.72e-05    0.4878  ██████████              0.675  ●●●●●●●●●●●●●●
    4    4.44e-05    0.2927  ██████                  0.723  ●●●●●●●●●●●●●●
    5    4.17e-05    0.1840  ████                    0.747  ●●●●●●●●●●●●●●●
    6    3.89e-05    0.1224  ███                     0.886  ●●●●●●●●●●●●●●●●●●
    7    3.61e-05    0.0829  ██                      0.950  ●●●●●●●●●●●●●●●●●●●
    8    3.33e-05    0.0714  █                       0.919  ●●●●●●●●●●●●●●●●●●
    9    3.06e-05    0.0602  █                       0.935  ●●●●●●●●●●●●●●●●●●●
   10    2.78e-05    0.0450  █                       0.936  ●●●●●●●●●●●●●●●●●●●
   11    2.50e-05    0.0318  █                       0.949  ●●●●●●●●●●●●●●●●●●●
   12    2.22e-05    0.0215                          0.942  ●●●●●●●●●●●●●●●●●●●
   13    1.94e-05    0.0

## 4. 채점 — 정답률 · 오답률

같은 시험 문장을 **규칙만**으로, 그리고 **규칙 + 방금 학습한 모델**로 채점해 나란히 본다.
- 등급 정답률: 예측 등급(L0~L3)이 정답과 같은 비율. 오답률 = 1 − 정답률
- 악성 재현율: 정답이 L2 이상인 문장을 L2 이상으로 잡은 비율 (놓치면 악성 채팅이 그대로 보인다)
- 정상 오탐률: 정답이 L0 인 문장을 L1 이상으로 본 비율 (높으면 멀쩡한 채팅을 가린다)
- 가릴 조각 정답률: 숨길 욕설 조각이 정답과 완전히 같은 비율 · 피드백 유무 정답률

In [6]:
def run_eval(model_path):
    os.environ[model.ENV_VAR] = str(model_path)
    impl.run(evaluate.to_request({"text": "준비"}, -1))  # 모델 로딩을 속도 측정에서 뺀다
    return {name: evaluate.evaluate(rows) for name, rows in test_rows.items()}


def pct(v):
    return "   -  " if v is None else f"{v * 100:5.1f}%"


results = {"규칙만": run_eval(resolve_repo_path("models/none")), "규칙 + 모델": run_eval(OUT)}
for name in test_rows:
    print(f"\n■ {name}")
    print(
        f"{'구성':<10} {'등급 정답률':>8} {'오답률':>7} {'악성 재현율':>8} {'정상 오탐률':>8} {'가릴 조각':>8} {'피드백 유무':>8}"
    )
    for label, res in results.items():
        r = res[name]
        print(
            f"{label:<10} {pct(r['level_accuracy']):>10} {pct(1 - r['level_accuracy']):>9} {pct(r['severe_recall']):>10} "
            f"{pct(r['clean_false_positive_rate']):>11} {pct(r['span_accuracy']):>10} {pct(r['feedback_accuracy']):>10}"
        )


■ 기존 시험 60문장
구성           등급 정답률     오답률   악성 재현율   정상 오탐률    가릴 조각   피드백 유무
규칙만             86.7%     13.3%      76.7%        0.0%      95.0%      98.3%
규칙 + 모델        100.0%      0.0%     100.0%        0.0%      95.0%     100.0%

■ 챔피언 이름 시험 43문장
구성           등급 정답률     오답률   악성 재현율   정상 오탐률    가릴 조각   피드백 유무
규칙만             65.1%     34.9%      44.4%        0.0%     100.0%      90.7%
규칙 + 모델        100.0%      0.0%     100.0%        0.0%     100.0%      93.0%

■ 협박 시험 40문장
구성           등급 정답률     오답률   악성 재현율   정상 오탐률    가릴 조각   피드백 유무
규칙만             57.5%     42.5%      19.0%        0.0%     100.0%     100.0%
규칙 + 모델         95.0%      5.0%      95.2%        5.3%     100.0%      95.0%


In [7]:
for name in test_rows:
    r = results["규칙 + 모델"][name]
    print(f"\n■ {name} — 유형별 (정밀도 / 재현율 / F1 / 정답 수), 규칙 + 모델")
    for label, m in r["harm_types"].items():
        print(f"  {label:<18} {pct(m['precision'])} {pct(m['recall'])} {pct(m['f1'])} {m['support']:>4}")
    print("  혼동 행렬 (행 = 정답, 열 = 예측 L0~L3)")
    for gold, row in r["confusion"].items():
        print(f"    L{gold} " + " ".join(f"{v:>4}" for v in row.values()))
    print(f"  틀린 문장 {len(r['errors'])}개")
    for e in r["errors"]:
        print(f"    정답 L{e['gold']} → 예측 L{e['pred']}  {e['text']}")


■ 기존 시험 60문장 — 유형별 (정밀도 / 재현율 / F1 / 정답 수), 규칙 + 모델
  profanity          100.0% 100.0% 100.0%   26
  family_insult      100.0% 100.0% 100.0%    6
  sexual_harassment   66.7% 100.0%  80.0%    2
  blame              100.0%  95.0%  97.4%   20
  instigation        100.0% 100.0% 100.0%    1
  threat             100.0% 100.0% 100.0%    1
  혼동 행렬 (행 = 정답, 열 = 예측 L0~L3)
    L0   21    0    0    0
    L1    0    9    0    0
    L2    0    0   21    0
    L3    0    0    0    9
  틀린 문장 0개

■ 챔피언 이름 시험 43문장 — 유형별 (정밀도 / 재현율 / F1 / 정답 수), 규칙 + 모델
  profanity          100.0% 100.0% 100.0%    6
  family_insult      100.0% 100.0% 100.0%    5
  sexual_harassment  100.0% 100.0% 100.0%    3
  blame               94.4%  94.4%  94.4%   18
  instigation        100.0% 100.0% 100.0%    2
  threat                -      -      -      0
  혼동 행렬 (행 = 정답, 열 = 예측 L0~L3)
    L0   14    0    0    0
    L1    0    2    0    0
    L2    0    0   19    0
    L3    0    0    0    8
  틀린 문장 0개

■ 협박 시험 40문장 — 유형별 (정밀도 /

## 5. 원본 → 숨김 문장

impl 이 실제로 내는 것: 가릴 조각(`profanity_spans`), 피드백 조각(`feedback_spans`), 유형, 등급.
여기서는 가릴 조각을 `*` 로 바꿔 숨김 문장을 만들어 보인다(앱에서는 정책표와 rewrite 가 화면 표시를 정한다).

In [8]:
os.environ[model.ENV_VAR] = str(OUT)
SAMPLES = [
    ("바론 전에 시야 잡고 모이자", None),
    ("정글 ㅅㅂ 전령 좀 풀어라", "정글 시발 전령 좀 풀어라"),
    ("원딜 딜량 서폿보다 낮네 ㅋㅋ", None),
    ("아 ㅈㄴ 아깝다 한 대 차이", "아 존나 아깝다 한 대 차이"),
    ("미드 병신아 로밍 갈 때 핑 좀 찍어", None),
    ("다들 탑 신고해 0/8 이다", None),
    ("서폿 니 엄마가 그렇게 가르쳤냐", None),
    ("우리 원딜 8/1 이니까 지켜주자", None),
    ("너 집 주소 안다 끝나고 찾아간다", None),
    ("상대 원딜 찾아가서 죽이자 혼자 있음", None),
]
for i, (text, norm) in enumerate(SAMPLES):
    hints = "lexicon:" + next(w for w in text.split() if w not in (norm or "")) if norm else ""
    out = impl.run(evaluate.to_request({"text": text, "text_norm": norm or text, "hints": hints}, i))
    masked = text
    for span in out.profanity_spans:
        masked = masked.replace(span, "*" * len(span))
    types = ",".join(t.value for t in out.harm_types) or "-"
    print(f"원본  {text}")
    print(f"숨김  {masked}")
    print(
        f"      L{int(out.level)} · 유형 {types} · 가릴 조각 {out.profanity_spans or '-'} · 피드백 {out.feedback_spans or '-'}\n"
    )

원본  바론 전에 시야 잡고 모이자
숨김  바론 전에 시야 잡고 모이자
      L0 · 유형 - · 가릴 조각 - · 피드백 -

원본  정글 ㅅㅂ 전령 좀 풀어라
숨김  정글 ** 전령 좀 풀어라
      L2 · 유형 profanity,blame · 가릴 조각 ['ㅅㅂ'] · 피드백 ['전령 좀 풀어라']

원본  원딜 딜량 서폿보다 낮네 ㅋㅋ
숨김  원딜 딜량 서폿보다 낮네 ㅋㅋ
      L2 · 유형 blame · 가릴 조각 - · 피드백 -

원본  아 ㅈㄴ 아깝다 한 대 차이
숨김  아 ** 아깝다 한 대 차이
      L1 · 유형 profanity · 가릴 조각 ['ㅈㄴ'] · 피드백 -

원본  미드 병신아 로밍 갈 때 핑 좀 찍어
숨김  미드 **아 로밍 갈 때 핑 좀 찍어
      L2 · 유형 profanity,blame · 가릴 조각 ['병신'] · 피드백 ['로밍 갈 때 핑 좀 찍어']

원본  다들 탑 신고해 0/8 이다
숨김  다들 탑 신고해 0/8 이다
      L2 · 유형 blame,instigation · 가릴 조각 - · 피드백 -

원본  서폿 니 엄마가 그렇게 가르쳤냐
숨김  서폿 니 엄마가 그렇게 가르쳤냐
      L3 · 유형 family_insult · 가릴 조각 - · 피드백 -



원본  우리 원딜 8/1 이니까 지켜주자
숨김  우리 원딜 8/1 이니까 지켜주자
      L0 · 유형 - · 가릴 조각 - · 피드백 -



원본  너 집 주소 안다 끝나고 찾아간다
숨김  너 집 주소 안다 끝나고 찾아간다
      L3 · 유형 threat · 가릴 조각 - · 피드백 -

원본  상대 원딜 찾아가서 죽이자 혼자 있음
숨김  상대 원딜 찾아가서 죽이자 혼자 있음
      L0 · 유형 - · 가릴 조각 - · 피드백 -



## 6. 응답 시간

기준: **1문장 0.5초 미만, 최대 1초 미만**. 모델을 처음 불러오는 시간은 따로 잰다(앱은 한 번만 불러온다).

In [9]:
model._load.cache_clear()
started = time.perf_counter()
impl.run(evaluate.to_request({"text": "준비"}, -1))
first = time.perf_counter() - started

rows = [r for rows in test_rows.values() for r in rows]
times = []
for _ in range(3):
    for i, r in enumerate(rows):
        req = evaluate.to_request(r, i)
        t = time.perf_counter()
        impl.run(req)
        times.append((time.perf_counter() - t) * 1000)
times.sort()
print(f"모델 첫 로딩 + 첫 문장: {first:.2f}초 (한 번만)")
print(
    f"문장 {len(times)}번 실행 · 평균 {statistics.fmean(times):.1f}ms · 중앙값 {statistics.median(times):.1f}ms"
    f" · p95 {times[int(0.95 * (len(times) - 1))]:.1f}ms · 최대 {times[-1]:.1f}ms"
)
print(f"0.5초 이상 {sum(t >= 500 for t in times)}번 · 1초 이상 {sum(t >= 1000 for t in times)}번")
rule_ms = results["규칙만"]["기존 시험 60문장"]["latency_ms"]["mean"]
print(f"참고: 규칙만 쓸 때 평균 {rule_ms:.2f}ms")

모델 첫 로딩 + 첫 문장: 0.12초 (한 번만)
문장 429번 실행 · 평균 26.0ms · 중앙값 26.0ms · p95 27.4ms · 최대 28.9ms
0.5초 이상 0번 · 1초 이상 0번
참고: 규칙만 쓸 때 평균 0.14ms


## 정리와 한계

- impl 은 **문장 하나**만 본다. 킬뎃을 들먹이는 조롱처럼 흐름을 봐야 하는 판단은 judge 가 한다.
- 시험 문장이 적고(60 + 43 + 40), 학습·시험 문장을 모두 AI 가 썼으며, 대상 규칙은 시험 오답을 보면서 고쳤다. **정답률은 실제보다 높게 나왔을 수 있다.**
- 응답 시간은 이 노트북을 돌린 PC(CPU) 기준이다. 실제 대상인 Windows PC 에서는 따로 재야 한다.
- 비난 표현은 "있다/없다"(유형)로만 판별한다. 가릴 **조각**을 내는 것은 욕설·패드립·성희롱 낱말뿐이다(비난 구절의 위치는 계약에 칸이 없다).